# ETIS transformer comparison (local runtime only)

This notebook fine-tunes MARBERTv2 and CAMeLBERT-DA on the frozen ETIS train/validation rows and writes validation predictions in the local-model JSON format. Connect this Colab notebook to a **local runtime** first. Hosted Colab is intentionally rejected: SMS text and URLs must never be uploaded to a remote runtime. Pretrained model weights may be downloaded from Hugging Face; dataset content and predictions stay local.

The repository root must be the current working directory and contain `evaluation/reports/2026-10-v3/text_split.json` and `text_trainval.csv`. It reads no test split and sends no messages or URLs to a service.

In [ ]:
import os, sys
if 'COLAB_RELEASE_TAG' in os.environ:
    raise RuntimeError('Hosted Colab is disabled for privacy. Connect to a local runtime; do not upload messages or URLs.')
from pathlib import Path
ROOT = Path.cwd()
REPORT = ROOT / 'evaluation/reports/2026-10-v3'
assert (REPORT / 'text_split.json').is_file() and (REPORT / 'text_trainval.csv').is_file(), 'Open this notebook from the ETIS repository root.'
print('Local runtime confirmed:', ROOT)

In [ ]:
# Install libraries in the local runtime. Model weights may be fetched; ETIS data are never transmitted.
%pip install -q 'transformers>=4.45,<5' 'torch>=2.2' 'accelerate>=0.34' 'pandas>=2.0' 'scikit-learn>=1.4' safetensors

In [ ]:
import csv, hashlib, json, random, time
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from torch.utils.data import Dataset
split = json.loads((REPORT/'text_split.json').read_text(encoding='utf-8'))
rows = pd.read_csv(REPORT/'text_trainval.csv', keep_default_na=False)
expected = {r['row_id']:r for r in split['rows']}
assert rows.id.astype(str).is_unique and set(rows.id.astype(str)) == set(expected), 'Row IDs differ from the frozen split'
assert set(rows.split) == {'train','val'}
for r in rows.itertuples():
    item = expected[str(r.id)]
    assert hashlib.sha256(str(r.text).encode()).hexdigest() == item['content_sha256']
    assert all(str(getattr(r, key)) == str(item[key]) for key in ['label','source','language','content_type','group_id','split']), 'Row metadata differs from frozen split'
train = rows[rows.split=='train'].reset_index(drop=True); valid = rows[rows.split=='val'].reset_index(drop=True)
SEEDS = [42, 43, 44]
MODELS = {'marbertv2':'UBC-NLP/MARBERTv2', 'camelbert_da':'CAMeL-Lab/bert-base-arabic-camelbert-da'}
OUT = REPORT/'predictions'; OUT.mkdir(exist_ok=True)
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()
class Rows(Dataset):
    def __init__(self, frame, tokenizer):
        self.enc=tokenizer(frame.text.astype(str).tolist(), truncation=True, max_length=256)
        self.labels=(frame.label=='phishing').astype(int).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self,i): return {**{k:torch.tensor(v[i]) for k,v in self.enc.items()},'labels':torch.tensor(self.labels[i])}
for model_name, model_id in MODELS.items():
  for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tok=AutoTokenizer.from_pretrained(model_id)
    model=AutoModelForSequenceClassification.from_pretrained(model_id,num_labels=2)
    args=TrainingArguments(output_dir=str(ROOT/'evaluation/local-results'/f'{model_name}-{seed}'),seed=seed, data_seed=seed, num_train_epochs=3, learning_rate=2e-5, per_device_train_batch_size=16, weight_decay=0.01, save_strategy='no', logging_strategy='epoch', report_to=[], fp16=torch.cuda.is_available(), use_cpu=not torch.cuda.is_available())
    trainer=Trainer(model=model,args=args,train_dataset=Rows(train,tok),data_collator=DataCollatorWithPadding(tok))
    trainer.train()
    model_dir=OUT/f'{model_name}_seed{seed}'; model_dir.mkdir(exist_ok=True)
    model.save_pretrained(model_dir,safe_serialization=True); tok.save_pretrained(model_dir)
    weights=model_dir/'model.safetensors'; model_sha=file_hash(weights) if weights.exists() else file_hash(model_dir/'pytorch_model.bin')
    model.to('cpu'); model.eval(); scores=[]; durations=[]
    warmup=tok(str(valid.text.iloc[0]),return_tensors='pt',truncation=True,max_length=256,padding=True)
    with torch.no_grad(): model(**warmup)
    for text in valid.text.astype(str):
      t=time.perf_counter()
      batch=tok(text,return_tensors='pt',truncation=True,max_length=256,padding=True)
      with torch.no_grad(): score=torch.softmax(model(**batch).logits,dim=-1)[0,1].item()
      durations.append((time.perf_counter()-t)*1000); scores.append(score)
    records=[{'row_id':str(r.id),'content_sha256':expected[str(r.id)]['content_sha256'],'label':int(r.label=='phishing'),'score':float(s)} for r,s in zip(valid.itertuples(),scores)]
    obj={'runtime':'local','input_policy':'first 256 tokenizer tokens per message; no sliding-window aggregation','model':model_name,'model_id':model_id,'seed':seed,'data_sha256':split['dataset_identity']['dataset_sha256'],'text_split_sha256':split['text_split_sha256'],'model_sha256':model_sha,'libraries':{'torch':torch.__version__,'transformers':__import__('transformers').__version__,'numpy':np.__version__,'pandas':pd.__version__},'model_size_bytes':sum(p.stat().st_size for p in model_dir.iterdir() if p.is_file()),'cpu_latency':{'batch_size':1,'p50_ms':float(np.percentile(durations,50)),'p95_ms':float(np.percentile(durations,95)),'n':len(durations),'cpu_only':True},'rows':records}
    (OUT/f'{model_name}_seed{seed}.json').write_text(json.dumps(obj,ensure_ascii=False,separators=(',',':')),encoding='utf-8')
    del model, trainer; torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('Saved six validation prediction files under', OUT)